In [14]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model

# -------------------- LOAD MODEL --------------------
model = load_model("best_gesture_model.h5")

# Your gesture labels (change based on your dataset)
GESTURE_CLASSES = ["0", "1", "2", "3", "4", "5"]

# -------------------- MEDIAPIPE SETUP --------------------
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# -------------------- LANDMARK PREPROCESSING --------------------
def extract_landmarks(hand_landmarks):
    """
    Convert MediaPipe hand landmarks into a normalized 63-length vector (x,y,z).
    """
    data = []
    for lm in hand_landmarks.landmark:
        data.append(lm.x)
        data.append(lm.y)
        data.append(lm.z)
    return np.array(data)

# -------------------- REAL-TIME LOOP --------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            # Extract 63 values
            landmarks = extract_landmarks(hand_landmarks)

            # Model expects shape (1,63)
            input_data = np.expand_dims(landmarks, axis=0)

            # Predict gesture
            prediction = model.predict(input_data, verbose=0)
            class_id = np.argmax(prediction)
            gesture_name = GESTURE_CLASSES[class_id]

            # Display prediction
            cv2.putText(frame, f"Gesture: {gesture_name}",
                        (10, 40), cv2.FONT_HERSHEY_SIMPLEX,
                        1.2, (0, 255, 0), 3)

    cv2.imshow("Gesture Recognition (MediaPipe)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


RuntimeError: ValidatedGraphConfig Initialization failed.
ImageToTensorCalculator: ; RET_CHECK failure (mediapipe/calculators/tensor/image_to_tensor_calculator.cc:155) ValidateOptionOutputDims(options) returned INTERNAL: ; RET_CHECK failure (./mediapipe/calculators/tensor/image_to_tensor_utils.h:136) options.has_output_tensor_float_range() || options.has_output_tensor_int_range() || options.has_output_tensor_uint_range()Output tensor range is required. 
ConstantSidePacketCalculator: ; RET_CHECK failure (mediapipe/calculators/core/constant_side_packet_calculator.cc:66) (cc->OutputSidePackets().NumEntries(kPacketTag))==(options.packet_size())Number of output side packets has to be same as number of packets configured in options.
ConstantSidePacketCalculator: ; RET_CHECK failure (mediapipe/calculators/core/constant_side_packet_calculator.cc:66) (cc->OutputSidePackets().NumEntries(kPacketTag))==(options.packet_size())Number of output side packets has to be same as number of packets configu

In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import tf2onnx

# Load Keras model
model = load_model("best_gesture_model.h5")

# ---- FIX FOR TF 2.12+ ----
# tf2onnx expects this attribute but TF removed it
if not hasattr(model, "output_names"):
    model.output_names = ["output"]
# --------------------------

# Define model input signature
spec = (tf.TensorSpec((1, 63), tf.float32, name="input"),)

# Convert to ONNX
output_path = "gesture_model.onnx"
model_proto, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=spec,
    output_path=output_path
)

print("ONNX model saved successfully:", output_path)


2025-12-02 10:10:18.147872: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-02 10:10:23.408632: I tensorflow/core/grappler/devices.cc:75] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
2025-12-02 10:10:23.472377: I tensorflow/core/grappler/devices.cc:75] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)


ONNX model saved successfully: gesture_model.onnx


In [15]:
pip install mediapipe==0.9.3.0

ERROR: Could not find a version that satisfies the requirement mediapipe==0.9.3.0 (from versions: 0.10.13, 0.10.14, 0.10.15, 0.10.18, 0.10.20, 0.10.21)
ERROR: No matching distribution found for mediapipe==0.9.3.0
Note: you may need to restart the kernel to use updated packages.
